# ADTomo two-grid pipeline
Run the numbered scripts first, then inspect the same forward calculation here.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from adtomo import ForwardGrid, VelocityModel, predict_phase_times

data_dir = Path('data')
data = torch.load(data_dir / 'model_true.pt', weights_only=True)
model = VelocityModel(**data, trainable=False)
stations = pd.read_csv(data_dir / 'stations.csv')
events = pd.read_csv(data_dir / 'events.csv', dtype={'event_id': str})
picks = pd.read_csv(data_dir / 'picks.csv', dtype={'event_id': str, 'station_id': str})

In [ ]:
station = stations.iloc[0]
station_picks = picks[(picks.station_id == station.station_id) & (picks.phase_type == 'P')].copy()
events_by_id = events.set_index('event_id', verify_integrity=True)
station_event_ids = list(pd.unique(station_picks.event_id))
station_events = events_by_id.loc[station_event_ids].reset_index()
station_lonlatdepth = torch.tensor([station.longitude, station.latitude, station.depth_km], dtype=torch.float64)
event_lonlatdepth = torch.tensor(station_events[['longitude', 'latitude', 'depth_km']].values, dtype=torch.float64)
grid = ForwardGrid(station_lonlatdepth, event_lonlatdepth, model, spacing=5.0)
catalog_event_index = {event_id: i for i, event_id in enumerate(events.event_id)}
grid_event_index = {event_id: i for i, event_id in enumerate(station_event_ids)}
catalog_event_indices = torch.tensor([catalog_event_index[event_id] for event_id in station_picks.event_id])
grid_event_indices = torch.tensor([grid_event_index[event_id] for event_id in station_picks.event_id])
event_dt = torch.zeros(len(events), dtype=torch.float64)
predict_phase_times(model, grid, 'P', event_dt[catalog_event_indices], event_indices=grid_event_indices)[:5]